# Finetune Support Ticket Classifier Qwen3

Please use a **free** Tesla T4 Colab GPU to run this!

And we'll be using the LLaMA-Factory visual UI framework to finetune a Qwen3 model for classifying support tickets.
Project homepage: https://github.com/hiyouga/LLaMA-Factory

---
## The Scenario

You're a senior engineer at a fast-growing IT services team. Your helpdesk handles a steady stream of human-written tickets every day — inbound across email, Slack, and your help portal. Today, every ticket touches a human before it reaches the right queue. A triage agent reads it, decides intent, and routes it manually. That's hours of human time daily, and it scales linearly with your user base.

You fix this by deploying a lightweight classifier at the front of the pipeline. Every ticket hits the model first:

```
Active Directory   → identity and access team, account and login workflows
Fileservice        → file share and network drive permissions
O365               → Outlook, Skype, Teams, OneDrive, mailbox issues
EOL                → server retirement and decommissioning tasks
Software           → application installs, updates, and app access
Computer-Services  → printers, scanners, drivers, and device support
Support general    → everything else that needs a human triage pass
```

The model never writes a response. It reads intent and fires a webhook. The rest of your tooling takes over.

**Why not a frontier model?**
At scale, a frontier model is overkill for a seven-class router. A finetuned 1.7B model like the one we build here runs on a single modest CPU instance for a few dollars a month, with latency in the tens of milliseconds. That's the actual business case for finetuning: not raw capability, but cost and latency at scale on a well-scoped task. (Those cost and latency figures are the target you would size the deployment against — this notebook measures *accuracy*, not throughput, so treat them as motivation rather than as results we report.)

**Why not keyword matching or regex?**
"I can't access the shared folder" — Fileservice or Active Directory? "Outlook keeps asking me to sign in" — O365 or Software? "The printer driver won't install and the scanner is offline" — both. Natural language is compositional and ambiguous. Rule-based routers fail on negation, multi-intent tickets, and phrasing variation. In practice, keyword routing misclassifies a meaningful slice of real support volume, which means your automation quietly generates more work than it saves.

A finetuned model learns intent from context. That's the difference between a brittle script and a reliable system component.

---

## Install Dependencies

In [ ]:
%cd /content/
%rm -rf LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
%ls
!pip install -e .[torch,bitsandbytes]

### Check GPU environment

In [ ]:
import torch
try:
  assert torch.cuda.is_available() is True
except AssertionError:
  print("Please set up a GPU before using LLaMA Factory: https://medium.com/mlearning-ai/training-yolov4-on-google-colab-316f8fff99c6")

---
## Configuration

Every constant the notebook uses lives in this one cell: model names, paths, the label set, and the two system prompts.

This matters more than it looks. The system prompt used to build the training data **must** be byte-identical to the one used at inference — if they drift, the model sees a different task at test time than the one it was trained on and accuracy silently degrades. Defining each prompt exactly once removes that failure mode.

`ADAPTER_DIR` is the only value you need to edit by hand, and only *after* training. Leave it as-is for now: you'll come back and set it to the Output Dir that LLaMA Board reports when your run finishes.

In [ ]:
import json
from pathlib import Path

# ── Model ─────────────────────────────────────────────────────────────────────
BASE_MODEL_NAME = "Qwen/Qwen3-1.7B-Base"

# ── Paths ─────────────────────────────────────────────────────────────────────
# Set ADAPTER_DIR *after* training, to the Output Dir shown in LLaMA Board.
ADAPTER_DIR = "/content/LLaMA-Factory/saves/Qwen3-1.7B-Base/lora/train_XXXX-XX-XX-XX-XX-XX"

MERGED_DIR          = "/content/qwen3_merged"
LLAMA_DATA_DIR      = "/content/LLaMA-Factory/data"
TRAIN_JSON_PATH     = f"{LLAMA_DATA_DIR}/TRAIN.json"
DATASET_INFO        = f"{LLAMA_DATA_DIR}/dataset_info.json"
TEST_CSV            = "/content/test_split.csv"
BASELINE_PRED_JSON  = "/content/baseline_preds.json"

# Where to find the source data, in priority order: local file, then repo raw URL,
# then interactive upload as a last resort.
CSV_CANDIDATES = [
    "support_tickets.csv",
    "/content/support_tickets.csv",
    "/content/drive/MyDrive/support_tickets.csv",
]
CSV_RAW_URL = (
    "https://raw.githubusercontent.com/The-Gen-Academy/"
    "5A-Fine-Tune-a-Support-Ticket-Router/main/support_tickets.csv"
)

# ── Labels ────────────────────────────────────────────────────────────────────
# This ordering is the canonical one. It is passed as `labels=` to every sklearn
# metric so that reports, the confusion matrix and the comparison chart all agree.
LABEL_TOKENS = [
    "Support general", "Fileservice", "O365", "EOL",
    "Software", "Active Directory", "Computer-Services",
]
LABELS = set(LABEL_TOKENS)

# Single letters for the zero-shot baseline (see the baseline section for why).
CHOICES      = "ABCDEFG"
CHOICE2LABEL = {ch: lbl for ch, lbl in zip(CHOICES, LABEL_TOKENS)}

# ── Prompts ───────────────────────────────────────────────────────────────────
# Used for training data generation AND fine-tuned inference. Do not fork this.
SYSTEM_PROMPT = (
    "You are an IT helpdesk ticket routing assistant. "
    "Given a support ticket, respond with exactly one of the following categories: "
    "Support general, Fileservice, O365, EOL, Software, Active Directory, Computer-Services."
)

# Used only for the zero-shot baseline.
BASE_SYSTEM_PROMPT = (
    "You are an IT helpdesk ticket routing assistant. "
    "Classify the ticket by responding with ONLY a single letter — nothing else:\n"
    + "\n".join(f"{ch}) {lbl}" for ch, lbl in CHOICE2LABEL.items())
)

# ── Eval ──────────────────────────────────────────────────────────────────────
TEST_SIZE        = 0.2   # held-out fraction; LLaMA Board carves its own val set from the rest
RANDOM_STATE     = 42
EVAL_BATCH_SIZE  = 7     # sequences per forward pass; raise for speed, lower on OOM


def require_adapter() -> Path:
    """Return ADAPTER_DIR as a validated Path, or explain exactly what's missing."""
    p = Path(ADAPTER_DIR)
    if "XXXX" in ADAPTER_DIR:
        raise ValueError(
            "ADAPTER_DIR is still the placeholder. Set it in the Configuration cell to "
            "the Output Dir that LLaMA Board reported for your training run, then re-run "
            "that cell."
        )
    if not p.is_dir():
        raise FileNotFoundError(f"Adapter folder not found: {ADAPTER_DIR!r}")
    if not (p / "adapter_config.json").exists():
        raise FileNotFoundError(f"No adapter_config.json in {ADAPTER_DIR!r}")
    return p


def load_test_split():
    """Reload the held-out split so downstream cells survive a kernel restart."""
    import pandas as pd
    if not Path(TEST_CSV).exists():
        raise FileNotFoundError(
            f"{TEST_CSV} missing — re-run the 'Prepare Support Ticket Dataset' cell."
        )
    return pd.read_csv(TEST_CSV)


def load_baseline_preds():
    """Reload baseline predictions saved by the baseline cell."""
    if not Path(BASELINE_PRED_JSON).exists():
        raise FileNotFoundError(
            f"{BASELINE_PRED_JSON} missing — re-run the baseline measurement cell."
        )
    with open(BASELINE_PRED_JSON, "r", encoding="utf-8") as f:
        d = json.load(f)
    return d["y_true"], d["y_pred_base"]


print(f"Base model  : {BASE_MODEL_NAME}")
print(f"Classes     : {len(LABEL_TOKENS)} -> {', '.join(LABEL_TOKENS)}")
print(f"Merged dir  : {MERGED_DIR}")
if "XXXX" in ADAPTER_DIR:
    print("\nADAPTER_DIR : not set yet (expected — set it after training).")
else:
    print(f"ADAPTER_DIR : {ADAPTER_DIR}")

---
## Prepare Support Ticket Dataset

This cell loads `support_tickets.csv`, creates a stratified train / held-out split, converts the training portion to **ShareGPT JSON format**, writes it to `/content/LLaMA-Factory/data/TRAIN.json`, and registers it in `dataset_info.json` so it appears in the LLaMA Board UI under the name `support_tickets`.

It finds the CSV on its own: a local copy first, then the course repo over HTTPS, and only if both fail does it prompt you to upload one. Nothing to click in the normal case.

### Why the held-out split is a *test* set, not a validation set

The terms get used loosely, but the distinction is the difference between an honest number and a flattering one. A **validation** set is what you look at while making decisions — how many epochs, what learning rate. A **test** set is what you touch once, at the end, to estimate real-world performance.

We only ever use this split to produce the final reported metrics, so it is a test set, and the notebook names it `test_split.csv`. LLaMA Board carves its own validation slice out of `TRAIN.json` for you (set **Validation size** to `0.1` in the Train tab) — that's what drives the eval-loss curve during training. Keeping the two separate is what makes the headline accuracy at the bottom of this notebook trustworthy.

In [ ]:
import json
import os
import urllib.request
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split


# ── 1. Locate the CSV ─────────────────────────────────────────────────────────
def resolve_csv() -> str:
    for cand in CSV_CANDIDATES:
        if Path(cand).exists():
            print(f"Found local CSV: {cand}")
            return cand

    dest = "/content/support_tickets.csv" if Path("/content").is_dir() else "support_tickets.csv"
    try:
        print(f"No local CSV — downloading from {CSV_RAW_URL}")
        urllib.request.urlretrieve(CSV_RAW_URL, dest)
        print(f"Downloaded -> {dest}")
        return dest
    except Exception as e:
        print(f"Download failed ({e}). Falling back to manual upload.")

    from google.colab import files
    print("Upload support_tickets.csv:")
    uploaded = files.upload()
    return list(uploaded.keys())[0]


CSV_PATH = resolve_csv()

# ── 2. Load + validate ────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH).rename(columns={"category_truth": "label"})

missing_cols = {"label", "text"} - set(df.columns)
if missing_cols:
    raise ValueError(f"CSV missing required column(s): {sorted(missing_cols)}")

n_raw = len(df)
df = df.dropna(subset=["label", "text"])
unknown = sorted(set(df["label"]) - LABELS)
if unknown:
    print(f"Dropping {len(unknown)} unrecognised label(s): {unknown}")
df = df[df["label"].isin(LABELS)]
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"\nLoaded {len(df):,} usable rows (from {n_raw:,} in file)")
print(df["label"].value_counts().to_string())

n_classes = df["label"].nunique()
if n_classes != len(LABEL_TOKENS):
    raise ValueError(f"Expected {len(LABEL_TOKENS)} classes, found {n_classes}")

# ── 3. Stratified train / held-out test split ────────────────────────────────
df_train, df_test = train_test_split(
    df, test_size=TEST_SIZE, stratify=df["label"], random_state=RANDOM_STATE,
)
df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)
print(f"\nTrain: {len(df_train):,} rows | Test (held out): {len(df_test):,} rows")
print("\nTrain label distribution:")
print(df_train["label"].value_counts().to_string())

# ── 4. Convert train split -> ShareGPT JSON ──────────────────────────────────
sharegpt_records = [
    {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": f"Support Ticket: {text}"},
            {"role": "assistant", "content": label},
        ]
    }
    for text, label in zip(df_train["text"], df_train["label"])
]

os.makedirs(LLAMA_DATA_DIR, exist_ok=True)
with open(TRAIN_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(sharegpt_records, f, indent=2, ensure_ascii=False)
print(f"\nWritten {len(sharegpt_records):,} records -> {TRAIN_JSON_PATH}")

# ── 5. Register dataset in dataset_info.json ─────────────────────────────────
info = {}
if Path(DATASET_INFO).exists():
    with open(DATASET_INFO, "r", encoding="utf-8") as f:
        info = json.load(f)

info["support_tickets"] = {
    "file_name": "TRAIN.json",
    "formatting": "sharegpt",
    "columns":   {"messages": "messages"},
    "tags": {
        "role_tag":      "role",
        "content_tag":   "content",
        "user_tag":      "user",
        "assistant_tag": "assistant",
        "system_tag":    "system",
    },
}

with open(DATASET_INFO, "w", encoding="utf-8") as f:
    json.dump(info, f, indent=2, ensure_ascii=False)
print(f"Registered 'support_tickets' in {DATASET_INFO}")

# ── 6. Save held-out split for evaluation ────────────────────────────────────
df_test.to_csv(TEST_CSV, index=False)
print(f"Test split saved -> {TEST_CSV}  ({len(df_test):,} rows)")

## Fine-tune model via LLaMA Board (INPUT REQUIRED)
1. Run this cell to start the LLaMA Board server. You should see a public and local URL in the output, open the public URL in your browser to access the UI.
2. Choose `Qwen/Qwen3-1.7B-Base` as the base model, and type `support_tickets` as the dataset. Make sure that you've set the training stage to **lora**.
3. Set **Validation size** to `0.1`. LLaMA Board will hold back 10% of `TRAIN.json` as its own validation set and plot an eval-loss curve alongside the training loss — that's what tells you whether the model is still learning or has started overfitting. The `test_split.csv` from the previous cell stays untouched until the final evaluation, so the numbers at the bottom of this notebook remain an honest estimate.
4. You can adjust the other hyperparameters as you see fit, but the defaults are good for a first run.
5. Once you finish training, take note of the **Output Dir** path shown at the bottom of this cell (e.g. `train_2026-...`) — go back to the **Configuration** cell, set `ADAPTER_DIR` to it, and re-run that cell.
6. Importantly, terminate this cell manually once training completes, as Colab won't know to stop it (this cell is technically a server daemon).

Note: To know that the training has completed, wait in the UI till you see a "training completed" notification on your screen or at the bottom of the training logs. Don't worry if you see a "syntax error" notification every now and then — this is a known issue with the LLaMA Board UI and doesn't affect training.

This will take a while (usually around 30-60 mins on a T4 GPU, depending on your model and hyperparameters), so feel free to grab a coffee while you wait!

### Hyperparameter Reference

The table below covers every parameter visible in the LLaMA Board Train tab. Use it to decide what to change if your first run underperforms.

Ordered by priority of tuning. Realistically, you only need to tune learning rate, epochs, batch size, and *maybe* LoRA rank. But you're perfectly fine using the defaults for a first run!

| Parameter | What it controls | Too high | Too low |
|-----------|-----------------|----------|---------|
| **Learning rate** | The step size the optimiser takes each update. Controls how aggressively the model updates toward each training example. | Loss spikes or diverges; model "unlearns" general knowledge (catastrophic forgetting) | Loss decreases very slowly; you need many more epochs to converge |
| **Epochs** | How many full passes over the training data the optimiser makes. | Model overfits — memorises training tickets and fails to generalise to new ones | Model underfits — loss is still falling when training stops; accuracy left on the table |
| **Batch size** | Number of samples processed together before each weight update. Larger batches give smoother gradient estimates but use more VRAM. | Runs out of VRAM; or gradients become too smooth and training stalls | Noisy gradient estimates; loss curve is jagged; training is slower wall-clock |
| **LoRA rank (r)** | Dimension of the low-rank matrices injected at each target layer. Higher rank = more trainable parameters = more expressive adapter. | More VRAM, slower training, risk of overfitting on small datasets | Adapter has too little capacity to capture the routing task; accuracy plateaus |
| **Gradient accumulation** | Simulates a larger batch by accumulating gradients over this many steps before updating. Effective batch = batch size × accumulation. | Effective batch becomes so large that small-dataset signal is washed out | Same as low batch size — noisy updates |
| **Cutoff length** | Maximum token length of each training example. Examples longer than this are truncated. | No downside except more VRAM and slower steps | Long tickets get silently truncated; model never sees the full context for complex tickets |
| **Max gradient norm** | Clips the gradient vector to this L2 norm before each update. Prevents a single bad batch from causing a catastrophically large weight update. | Gradients are never clipped; a noisy batch can cause a loss spike | Gradients are clipped too aggressively; effective learning rate is much lower than set; training slows |
| **LR scheduler** | Controls how the learning rate changes over training. `cosine` warms up then smoothly decays to near zero. `linear` decays linearly. `constant` never changes. | N/A — this is a choice, not a magnitude | N/A |
| **Compute type** | Numerical precision during training. `bf16` is standard on modern GPUs (A100, H100); `fp16` for older hardware (T4); `fp32` uses full precision but 2× the VRAM. | N/A — match to your GPU | Using `fp32` on a T4 will likely OOM |



In [ ]:
%cd /content/LLaMA-Factory/
!GRADIO_SHARE=1 llamafactory-cli webui

---
## Review Training — Loss Curve

Before merging the adapter, it's worth checking whether training actually converged. The loss curve is the most direct signal: it should fall steadily over steps and level off, not stay flat or oscillate. A curve that never meaningfully drops usually means the learning rate was too high, the dataset was too small, or the wrong chat template was used during data prep.

If you set **Validation size** in LLaMA Board, the eval loss is plotted too. Watch for the classic overfitting signature: training loss still falling while eval loss turns back upward. The step where eval loss bottoms out is the number of epochs you actually wanted.

This cell reads `ADAPTER_DIR` from the **Configuration** cell — set it there if you haven't already.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

adapter_path = require_adapter()

log_file = adapter_path / "trainer_log.jsonl"
if not log_file.exists():
    raise FileNotFoundError(f"trainer_log.jsonl not found in {ADAPTER_DIR!r}")

records = [json.loads(l) for l in log_file.read_text().splitlines() if l.strip()]

train_steps  = [r["current_steps"] for r in records if r.get("loss") is not None]
train_losses = [r["loss"]          for r in records if r.get("loss") is not None]
eval_steps   = [r["current_steps"] for r in records if r.get("eval_loss") is not None]
eval_losses  = [r["eval_loss"]     for r in records if r.get("eval_loss") is not None]

if not train_losses:
    raise ValueError(f"No loss entries found in {log_file} — did training actually run?")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_steps, train_losses, linewidth=1.5, color="#1565C0", alpha=0.85, label="train loss")
if eval_losses:
    ax.plot(eval_steps, eval_losses, linewidth=1.5, color="#E65100", marker="o",
            markersize=3, alpha=0.9, label="eval loss")
    best = min(range(len(eval_losses)), key=lambda i: eval_losses[i])
    ax.axvline(eval_steps[best], color="#E65100", linestyle="--", linewidth=0.8)
    ax.legend()
else:
    ax.legend()
ax.set_xlabel("Step")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Training loss curve")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("/content/training_curve.png", dpi=120)
plt.show()

print(f"Starting loss : {train_losses[0]:.4f}")
print(f"Final loss    : {train_losses[-1]:.4f}")
print(f"Total drop    : {train_losses[0] - train_losses[-1]:.4f}")
if eval_losses:
    print(f"Best eval loss: {min(eval_losses):.4f} at step {eval_steps[best]}")
    if eval_losses[-1] > min(eval_losses) * 1.05:
        print("Eval loss rose after its minimum — the run overfit; fewer epochs would do better.")
print()
if train_losses[-1] < 0.5:
    print("Loss is low — model has likely converged well.")
elif train_losses[-1] < 1.2:
    print("Loss is moderate — model has learned but may benefit from more epochs.")
else:
    print("Loss is still high — consider more epochs, a lower learning rate, or checking the data format.")

---
## How We Measure: Constrained Label Scoring

Both models in this notebook — the untouched base model and the fine-tuned one — are scored the exact same way. That symmetry is what makes the final comparison meaningful, so it's worth understanding before we load anything.

### The naive approach, and why it misleads

The obvious way to evaluate a classifier built on a language model is to let it generate text and then read the output:

```python
out = model.generate(**inputs, max_new_tokens=10)
label = match_against_known_labels(decode(out))   # and if nothing matches?
```

That last line is the problem. When generation produces something that isn't a label — an explanation, a lowercase variant, an empty string — you have to guess. Defaulting to one class quietly poisons that class's precision and hides the failure entirely. And it penalises the base model twice: once for not knowing the task, and again for not knowing the *output format*. The delta you'd report at the end would be a blend of "learned to route tickets" and "learned to reply with a bare label," which are very different achievements.

### What we do instead

We never generate. For each ticket we build the prompt, then ask the model to score all seven labels as *candidate continuations* and pick the highest. Concretely, for each candidate label we compute the sum of token log-probabilities of that label given the prompt, teacher-forced in a single forward pass:

```
score(label) = Σ log P(token_i | prompt, token_<i>)
```

Three properties fall out of this, all of which we want:

- **The prediction is always one of the seven classes.** There is no parse step, so there is no fallback branch and no silent corruption of any class.
- **Confidence is a real posterior.** Softmax over the seven scores gives a proper distribution across exactly the candidate set we care about, rather than a probability smeared over a 150k-token vocabulary.
- **It's fast.** One batched forward pass per ticket instead of up to ten sequential decode steps.

One caveat to be aware of: summed log-probability is a length-biased quantity — a label that tokenizes into six pieces accumulates six negative terms while `EOL` accumulates two or three. This is the mathematically correct decision rule (it maximises sequence likelihood over the candidate set) and a converged model overwhelms the bias easily, since it assigns near-zero log-prob per token to the right label. But if you ever see a systematic pull toward the shortest class name, length normalisation is the first thing to try.

The helpers below take `model` and `tokenizer` as arguments rather than reading globals, so the same code path serves the base model, the merged model, and any checkpoint you want to compare later.

In [ ]:
import torch
from tqdm.auto import tqdm


def build_prompt(tokenizer, ticket_text: str, system_prompt: str) -> str:
    """Render a ticket into the chat format the model was trained on."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": f"Support Ticket: {ticket_text}"},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def _pad_batch(seqs, pad_id, device):
    """Right-pad a list of token-id lists into a rectangular batch.

    Right padding is safe here because attention is causal and we only ever read
    logits at positions strictly inside the real sequence — padding sits after
    everything we look at and cannot influence it.
    """
    maxlen = max(len(s) for s in seqs)
    ids  = torch.full((len(seqs), maxlen), pad_id, dtype=torch.long)
    attn = torch.zeros((len(seqs), maxlen), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, : len(s)] = torch.tensor(s, dtype=torch.long)
        attn[i, : len(s)] = 1
    return ids.to(device), attn.to(device)


def score_completions(model, tokenizer, prompts, completions, batch_size=None):
    """Teacher-forced log-likelihood of each completion, for each prompt.

    Returns a list (one entry per prompt) of lists (one score per completion).
    """
    batch_size = batch_size or EVAL_BATCH_SIZE
    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id

    comp_ids = [tokenizer(c, add_special_tokens=False)["input_ids"] for c in completions]
    if any(len(c) == 0 for c in comp_ids):
        raise ValueError("A completion tokenized to zero tokens — check the label strings.")

    # The chat template already inserts the model's special tokens, so the prompt
    # must be tokenized with add_special_tokens=False to avoid a duplicate BOS.
    rows = []
    for pi, p in enumerate(prompts):
        p_ids = tokenizer(p, add_special_tokens=False)["input_ids"]
        for ci, c_ids in enumerate(comp_ids):
            rows.append((pi, ci, p_ids + c_ids, len(p_ids), len(c_ids)))

    out = [[0.0] * len(completions) for _ in prompts]

    for start in range(0, len(rows), batch_size):
        chunk = rows[start : start + batch_size]
        ids, attn = _pad_batch([r[2] for r in chunk], pad_id, model.device)
        with torch.no_grad():
            logits = model(input_ids=ids, attention_mask=attn).logits

        for k, (pi, ci, _seq, plen, clen) in enumerate(chunk):
            # Token at position plen+j is predicted by the logits at plen+j-1.
            # Slice to just those rows before softmaxing — the full [T, vocab]
            # tensor in float32 would be hundreds of MB per batch.
            sel = logits[k, plen - 1 : plen - 1 + clen, :].float()
            tgt = ids[k, plen : plen + clen]
            out[pi][ci] = torch.log_softmax(sel, dim=-1).gather(
                1, tgt.unsqueeze(1)
            ).sum().item()

    return out


def predict(model, tokenizer, texts, system_prompt, completions, mapping=None,
            desc="Scoring", batch_size=None, show_progress=True):
    """Classify an iterable of ticket texts. Returns (labels, confidences).

    `completions` are the strings actually scored; `mapping` optionally converts
    each one to its final label (used by the A-G baseline).
    """
    texts  = list(texts)
    labels, confs = [], []
    iterator = tqdm(texts, desc=desc) if show_progress else texts

    for text in iterator:
        prompt = build_prompt(tokenizer, text, system_prompt)
        scores = score_completions(model, tokenizer, [prompt], completions,
                                   batch_size=batch_size)[0]
        probs  = torch.softmax(torch.tensor(scores), dim=-1).tolist()
        best   = max(range(len(scores)), key=lambda i: scores[i])
        chosen = completions[best]
        labels.append(mapping[chosen] if mapping else chosen)
        confs.append(probs[best])

    return labels, confs


def check_chat_template(tokenizer, source: str) -> None:
    """Fail loudly if the tokenizer has no chat template.

    Qwen3-*-Base is a *base* checkpoint and ships without one. LLaMA-Factory saves
    a tokenizer that does have the training template into the adapter directory,
    which is why we prefer that source.
    """
    if getattr(tokenizer, "chat_template", None):
        return
    raise ValueError(
        f"Tokenizer loaded from {source!r} has no chat_template, so prompts cannot be "
        "rendered the way the model was trained.\n"
        f"Load the tokenizer from ADAPTER_DIR instead ({ADAPTER_DIR}) — LLaMA-Factory "
        "saves it there with the training template attached."
    )


print("Scoring helpers defined: build_prompt, score_completions, predict, check_chat_template")

---
## Measure the Baseline (Before Touching the Adapter)

### Why measure here, and why it's free

To know what fine-tuning actually contributed, we need a **control**: the same base model, on the same held-out tickets, with no task-specific training. We take that measurement now, before the adapter is attached, because the base model is already about to be loaded into GPU memory. Running baseline inference at this point costs one pass over the test set and nothing else — measuring it later would mean reloading a second copy of the weights.

### Why a letter-choice prompt for the baseline

The base model has never been told that `"Active Directory"` is the expected surface form. Asked to route a ticket, it might reasonably answer `"AD"`, `"active directory"`, `"the identity team"`, or a paragraph of reasoning. Scoring those seven long label strings directly would measure how well the base model guesses our formatting conventions, not how well it understands the tickets.

So we present the classes as seven lettered options and score the single tokens `A` through `G`. Single letters are unambiguous, are one token in any tokenizer, and are a format any instruction-following model handles. This gives the base model its best realistic shot.

Note what has *not* changed between baseline and fine-tuned: the scoring mechanism. Both use the constrained log-likelihood comparison from the previous cell — the base model over `A`–`G`, the fine-tuned model over the label strings it was actually trained to emit. Each model is scored on the format it is best equipped for, with identical machinery, so the delta we report at the end reflects learned routing ability rather than a change in output convention.

In [ ]:
import json

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import classification_report

adapter_path = require_adapter()

# ── Device ────────────────────────────────────────────────────────────────────
HAS_CUDA = torch.cuda.is_available()
HAS_MPS  = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
DEVICE   = "cuda" if HAS_CUDA else ("mps" if HAS_MPS else "cpu")
DTYPE    = torch.float16 if DEVICE in ("cuda", "mps") else torch.float32
print(f"Device: {DEVICE}")

# ── Tokenizer (prefer the adapter's — it carries the training chat template) ──
tok_src = str(adapter_path) if (adapter_path / "tokenizer_config.json").exists() else BASE_MODEL_NAME
_tokenizer = AutoTokenizer.from_pretrained(tok_src)
check_chat_template(_tokenizer, tok_src)
print(f"Tokenizer: {tok_src}")

# ── Load base model ──────────────────────────────────────────────────────────
print("\nLoading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, torch_dtype=DTYPE, device_map=DEVICE,
)
base_model.eval()

# ── Baseline inference on the held-out split ─────────────────────────────────
df_test = load_test_split()
y_true  = df_test["label"].tolist()

y_pred_base, _ = predict(
    base_model, _tokenizer, df_test["text"],
    system_prompt=BASE_SYSTEM_PROMPT,
    completions=list(CHOICES),
    mapping=CHOICE2LABEL,
    desc="Baseline inference",
)

with open(BASELINE_PRED_JSON, "w", encoding="utf-8") as f:
    json.dump({"y_true": y_true, "y_pred_base": y_pred_base}, f)
print(f"\nBaseline predictions saved -> {BASELINE_PRED_JSON}")

print("\n=== Baseline (no fine-tuning) ===")
print(classification_report(
    y_true, y_pred_base,
    labels=LABEL_TOKENS, target_names=LABEL_TOKENS,
    digits=3, zero_division=0,
))

---
## Merge the Adapter Weights

### What is a LoRA adapter and why do we need to merge it?

During fine-tuning, the original model weights were **frozen**. Instead of updating the full weight matrices (billions of parameters), LoRA injects small, trainable rank-decomposition matrices alongside the frozen ones. Only those lightweight matrices — a tiny fraction of the total parameter count — were updated during training.

After training you therefore have two separate things on disk:
- The original **base model** weights (unchanged)
- The **LoRA adapter** — a small set of delta weights that encode everything the model learned

For training this separation is efficient. For inference it adds overhead: the model must apply the adapter at every forward pass. **Merging** folds the adapter deltas mathematically back into the base weights, producing a single standalone model that is identical in behaviour to the adapted one but has no runtime overhead and can be used anywhere that accepts a standard model.

This cell attaches the adapter to the base model that's already in memory, merges it, and writes the result to `MERGED_DIR`. Because it saves to disk, the rest of the notebook can recover from a kernel restart without re-running any of the steps above — the next cell loads from `MERGED_DIR` if it's there.

Merging LoRA adapters ("writing model shards...") will take a while for larger models.

In [ ]:
import gc
import os
from pathlib import Path

import torch
import safetensors.torch as st
from peft import PeftConfig, get_peft_model
from peft.utils import set_peft_model_state_dict

adapter_path = require_adapter()

# ── Attach LoRA + load weights ───────────────────────────────────────────────
print("Attaching LoRA adapter...")
peft_cfg   = PeftConfig.from_pretrained(str(adapter_path))
peft_model = get_peft_model(base_model, peft_cfg)

candidates = [
    adapter_path / "adapter_model.safetensors",
    adapter_path / "adapters.safetensors",
    adapter_path / "adapter_model.bin",
]
weights_file = next((p for p in candidates if p.exists()), None)
if weights_file is None:
    raise FileNotFoundError(f"No adapter weights in {ADAPTER_DIR}")

if weights_file.suffix == ".safetensors":
    adapter_weights = st.load_file(str(weights_file), device=DEVICE)
else:
    adapter_weights = torch.load(str(weights_file), map_location=DEVICE)

load_result = set_peft_model_state_dict(peft_model, adapter_weights)
unexpected = getattr(load_result, "unexpected_keys", None)
if unexpected:
    print(f"Warning: {len(unexpected)} unexpected key(s) in adapter weights")

# ── Merge + save to disk ─────────────────────────────────────────────────────
print("Merging (this may take a while) ...")
_model = peft_model.merge_and_unload()

os.makedirs(MERGED_DIR, exist_ok=True)
_model.save_pretrained(MERGED_DIR)
_tokenizer.save_pretrained(MERGED_DIR)
stale = Path(MERGED_DIR) / "adapter_config.json"
if stale.exists():
    stale.unlink()
print(f"Saved -> {MERGED_DIR}")

# ── Drop the PEFT wrapper. Note merge_and_unload() returns the *same* module
#    object as base_model with the deltas folded in, so `_model is base_model`
#    and deleting the name frees nothing — the weights live on in `_model`.
del peft_model, adapter_weights, base_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

_model.eval()
print("Merged model ready.")

### Load the model to evaluate

This cell is the notebook's recovery point. If `_model` is already in memory from the merge above, it's used as-is. Otherwise the merged checkpoint is loaded from `MERGED_DIR`, which means you can restart the kernel, run the **Configuration** cell, and jump straight here without repeating the download, the baseline pass, or the merge.

In [ ]:
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if "_model" in globals() and _model is not None:
    print("Using the merged model already in memory.")
else:
    if not Path(MERGED_DIR).exists():
        raise FileNotFoundError(
            f"{MERGED_DIR} not found — run the merge cell above first."
        )
    HAS_CUDA = torch.cuda.is_available()
    HAS_MPS  = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    DEVICE   = "cuda" if HAS_CUDA else ("mps" if HAS_MPS else "cpu")
    DTYPE    = torch.float16 if DEVICE in ("cuda", "mps") else torch.float32

    print(f"Loading merged model from {MERGED_DIR} (device={DEVICE})...")
    _model = AutoModelForCausalLM.from_pretrained(
        MERGED_DIR, torch_dtype=DTYPE, device_map=DEVICE,
    )
    _model.eval()
    _tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
    check_chat_template(_tokenizer, MERGED_DIR)

# Restore evaluation state if this is a fresh kernel.
if "df_test" not in globals():
    df_test = load_test_split()
if "y_true" not in globals() or "y_pred_base" not in globals():
    y_true, y_pred_base = load_baseline_preds()

print(f"Ready. Test set: {len(df_test):,} rows | baseline preds: {len(y_pred_base):,}")

---
## Define `classify()` and Smoke Test the Merged Model

`classify()` is the function you would actually ship: ticket text in, route and confidence out. It's a thin wrapper over the constrained scoring helpers defined earlier — the seven candidates are now the full label strings, because the fine-tuned model was trained to emit exactly those.

The confidence it returns is the softmax probability of the winning label across the seven candidates. In production that number is the useful half of the output: route high-confidence tickets automatically, and send anything below a threshold (~0.85 is a reasonable starting point) to a human triage queue. Tuning that threshold trades automation rate against error rate, and the right value depends on how expensive a misroute is for you.

### Why a smoke test before the full evaluation

The five smoke-test tickets cover five of the seven classes with deliberately unambiguous phrasing. If any of them are wrong, something is broken — the wrong checkpoint was loaded, the chat template doesn't match the one used for training, or the run diverged. It takes seconds and it saves you from waiting out a full evaluation pass over a model that was never going to work.

In [ ]:
def classify(ticket_text: str, compute_confidence: bool = True):
    """Route one support ticket. Returns (label, confidence).

    Always returns one of LABEL_TOKENS — scoring is constrained to the label set,
    so there is no parse failure and no fallback class.
    """
    labels, confs = predict(
        _model, _tokenizer, [ticket_text],
        system_prompt=SYSTEM_PROMPT,
        completions=LABEL_TOKENS,
        show_progress=False,
    )
    return labels[0], (confs[0] if compute_confidence else 1.0)


# ── Smoke test ────────────────────────────────────────────────────────────────
smoke_tickets = [
    ("Please create a new user account for the new employee starting Monday.",    "Active Directory"),
    ("I cannot access the shared network folder — I keep getting Access Denied.", "Fileservice"),
    ("Outlook keeps asking me to re-enter my password and Teams won't sync.",     "O365"),
    ("We're ready to retire [SERVER] — please remove it from monitoring.",        "EOL"),
    ("Adobe Acrobat fails to install, error code 1603.",                          "Software"),
]

n_ok = 0
print(f"{'Ticket (truncated)':<60} {'Expected':<20} {'Predicted':<20} {'Conf':>6}")
print("-" * 110)
for ticket, expected in smoke_tickets:
    label, conf = classify(ticket)
    ok = label == expected
    n_ok += ok
    print(f"{ticket[:58]:<60} {expected:<20} {label:<20} {conf:>6.1%}  {'OK' if ok else '--'}")

print(f"\nSmoke test: {n_ok}/{len(smoke_tickets)} correct")
if n_ok < 4:
    print("Fewer than 4/5 correct — stop and check the checkpoint and chat template "
          "before running the full evaluation.")

---
## Evaluate the Fine-tuned Model on the Held-out Split

`test_split.csv` was held out before training — the model has never seen these tickets, and neither did LLaMA Board's validation set. Running inference over them gives an honest estimate of real-world routing performance.

Every prediction is one of the seven labels by construction, so this loop can't fail to parse and there are no rows to discard. The cell reports throughput at the end; if you have VRAM headroom on your T4, raising `EVAL_BATCH_SIZE` in the **Configuration** cell will speed it up, and lowering it is the fix if you hit an out-of-memory error.

In [ ]:
import time

if "df_test" not in globals():
    df_test = load_test_split()
if "y_true" not in globals():
    y_true, y_pred_base = load_baseline_preds()

t0 = time.perf_counter()
y_pred, y_conf = predict(
    _model, _tokenizer, df_test["text"],
    system_prompt=SYSTEM_PROMPT,
    completions=LABEL_TOKENS,
    desc="Fine-tuned inference",
)
elapsed = time.perf_counter() - t0

assert len(y_pred) == len(y_true), f"{len(y_pred)} predictions vs {len(y_true)} labels"
assert set(y_pred) <= set(LABEL_TOKENS), "a prediction escaped the label set"

print(f"\nEvaluated {len(y_pred):,} tickets in {elapsed:,.1f}s "
      f"({elapsed / len(y_pred) * 1000:.0f} ms/ticket)")
print(f"Mean confidence: {sum(y_conf) / len(y_conf):.1%}")

low_conf = sum(c < 0.85 for c in y_conf)
print(f"Below 0.85 confidence: {low_conf:,} tickets ({low_conf / len(y_conf):.1%}) "
      "— these are the ones you'd route to human triage.")

In [ ]:
from sklearn.metrics import classification_report

# labels= pins the class order to LABEL_TOKENS so that target_names lines up with
# it. Without labels=, sklearn orders classes alphabetically and applies
# target_names positionally, silently attributing every row to the wrong class.
print(classification_report(
    y_true, y_pred,
    labels=LABEL_TOKENS, target_names=LABEL_TOKENS,
    digits=3, zero_division=0,
))

# A few individual predictions, including the model's confidence.
print()
for i in [0, 10, 20, 30]:
    if i >= len(df_test):
        continue
    row  = df_test.iloc[i]
    pred, conf = classify(row["text"])
    flag = "" if pred == row["label"] else "   <-- MISROUTED"
    print(f"=== Ticket {i}{flag}")
    print(f"TRUE label: {row['label']}")
    print(f"PRED label: {pred}  (conf {conf:.1%})")
    print(f"TEXT      : {row['text'][:160]}")
    print()

### Reading the classification report

The report shows four numbers per class:

| Metric | What it measures |
|--------|-----------------|
| **Precision** | Of all tickets the model routed to this class, what fraction actually belonged there — measures false alarm rate |
| **Recall** | Of all tickets that truly belong to this class, what fraction did the model catch — measures miss rate |
| **F1** | Harmonic mean of precision and recall — a single balanced score per class |
| **Support** | Number of validation tickets in this class |

### Routing cost asymmetry

Overall accuracy treats every mistake equally. In a real helpdesk, they are not equal.

An `Active Directory` ticket misrouted means a new employee cannot log in on their first day — high urgency, immediately visible to the business. A `Fileservice` ticket in the general queue delays access to shared files — serious, but less acute. A `Support general` ticket misrouted wastes a specialist's time, but no critical system is blocked.

**The metric that matters for production: per-class recall on high-urgency categories.**

A model with 90% overall accuracy but 60% `Active Directory` recall is routing 40% of account-provisioning requests to the wrong team. A model with 83% overall accuracy and 95% `Active Directory` recall is safer, even though it looks worse on the headline number. This is why the confusion matrix and per-class F1 are more informative than a single accuracy figure for any triage or routing problem.

### Confusion matrix and error analysis

The confusion matrix shows **where the model makes mistakes**, not just how many. Each row is a true class; each column is what the model predicted. The diagonal is correct predictions; everything off-diagonal is an error.

The colour is row-normalised (each row sums to 1.0), so you can immediately read recall per class — a dark diagonal cell means high recall. The raw counts are printed inside each cell so you can judge statistical significance: a single confused ticket out of eight is less meaningful than five out of eight.

Below the matrix the cell does the reading for you: it ranks the worst confusion pairs, prints the per-class recall table sorted worst-first, and shows the actual text of tickets from the single worst pair. That last part is the most useful output in the notebook — a systematic confusion is either a data problem (those two classes genuinely overlap in how users phrase them, and your label taxonomy is leaky) or a coverage problem (too few training examples to separate them). You can usually tell which within a few examples.

Note that every metric here is computed with `labels=LABEL_TOKENS`, the same ordering used in the classification report above, so the two are directly comparable — the report's recall column equals this matrix's normalised diagonal.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm      = confusion_matrix(y_true, y_pred, labels=LABEL_TOKENS)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm_norm, annot=cm, fmt="d", cmap="Blues",
    xticklabels=LABEL_TOKENS, yticklabels=LABEL_TOKENS,
    linewidths=0.5, ax=ax, vmin=0, vmax=1,
)
ax.set_title("Confusion matrix (counts shown, colour = row-normalised recall)")
ax.set_ylabel("True label")
ax.set_xlabel("Predicted label")
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("/content/confusion_matrix.png", dpi=120)
plt.show()

# ── Per-class recall, worst first ────────────────────────────────────────────
recalls = sorted(
    ((LABEL_TOKENS[i], cm_norm[i, i], int(cm[i].sum())) for i in range(len(LABEL_TOKENS))),
    key=lambda t: t[1],
)
print(f"{'Class':<20} {'Recall':>8} {'Support':>8}")
print("-" * 38)
for name, rec, support in recalls:
    flag = "  <-- below 0.85" if rec < 0.85 else ""
    print(f"{name:<20} {rec:>7.1%} {support:>8,}{flag}")

# ── Top confusion pairs ──────────────────────────────────────────────────────
pairs = [
    (LABEL_TOKENS[i], LABEL_TOKENS[j], int(cm[i, j]), cm[i, j] / max(cm[i].sum(), 1))
    for i in range(len(LABEL_TOKENS))
    for j in range(len(LABEL_TOKENS))
    if i != j and cm[i, j] > 0
]
pairs.sort(key=lambda p: p[2], reverse=True)

print(f"\nTop confusion pairs ({len(pairs)} non-empty off-diagonal cells):")
if not pairs:
    print("  none — no misclassifications on the held-out split.")
for true_lbl, pred_lbl, n, frac in pairs[:5]:
    print(f"  {true_lbl:<20} -> {pred_lbl:<20} {n:>4} tickets ({frac:.1%} of that class)")

# ── Sample tickets from the worst pair ───────────────────────────────────────
if pairs:
    worst_true, worst_pred, n, _ = pairs[0]
    print(f"\nExample tickets: true '{worst_true}' predicted '{worst_pred}' ({n} total)")
    print("-" * 100)
    shown = 0
    for idx in range(len(df_test)):
        if y_true[idx] == worst_true and y_pred[idx] == worst_pred:
            print(f"  [conf {y_conf[idx]:.0%}] {df_test.iloc[idx]['text'][:150]}")
            shown += 1
            if shown >= 5:
                break

---
## Baseline vs Fine-tuned Comparison

### What "baseline" means here

The baseline is the same model weights that fine-tuning started from, evaluated on the same held-out tickets, with **zero task-specific training**. It represents what the model knows purely from pre-training — general language understanding, instruction-following, some domain knowledge absorbed from internet text — with no exposure to your labelled examples.

Both sides were measured with the same constrained-scoring machinery (see *How We Measure* above), each over the candidate format best suited to it: `A`–`G` for the base model, the label strings for the fine-tuned one. Neither model was ever asked to produce free text that we then had to parse, so no part of the gap below is an artefact of one model happening to guess our output conventions.

### What the delta tells you

The gap between baseline and fine-tuned accuracy is the direct, measurable value of the labelled training data and the fine-tuning run. A large delta on classes that were near-random in the baseline confirms the model genuinely learned task-specific routing signal, rather than pattern-matching something it already knew.

Bear in mind the floor: with seven balanced classes, random guessing scores about 14%. A baseline near that number means the base model had essentially no usable prior for this taxonomy.

### Reading the comparison chart

Each class has two bars: light blue for the base model, dark blue for the fine-tuned model. The rightmost pair shows overall accuracy.

Things to look for:

- **Classes where the base model scores near zero** — the base model has no idea how to route those tickets; fine-tuning taught it everything it knows about them
- **Classes where the gap is small** — the base model already had strong signal, often for classes whose names map closely onto everyday language, like `Software`
- **Classes where the fine-tuned model regressed** — rare, but possible if that class was underrepresented in training or if its labels overlap another class
- **The overall accuracy delta** — the headline number printed below the chart; typical values after a short LoRA run on a few thousand examples land in the +40–60 percentage point range

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, classification_report


def per_class_f1(y_t, y_p, labels):
    # labels= is essential: it fixes the class order so each dict key refers to
    # the class it names, rather than to sklearn's alphabetical position.
    report = classification_report(
        y_t, y_p, labels=labels, target_names=labels,
        output_dict=True, zero_division=0,
    )
    return {lbl: report[lbl]["f1-score"] for lbl in labels}


ft_f1    = per_class_f1(y_true, y_pred,      LABEL_TOKENS)
base_f1  = per_class_f1(y_true, y_pred_base, LABEL_TOKENS)
ft_acc   = accuracy_score(y_true, y_pred)
base_acc = accuracy_score(y_true, y_pred_base)

labels_plot = LABEL_TOKENS + ["overall accuracy"]
ft_vals     = [ft_f1[l]   for l in LABEL_TOKENS] + [ft_acc]
base_vals   = [base_f1[l] for l in LABEL_TOKENS] + [base_acc]

x     = np.arange(len(labels_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
bars_base = ax.bar(x - width/2, base_vals, width, label="Base model (no fine-tuning)",
                   color="#90CAF9", edgecolor="white")
bars_ft   = ax.bar(x + width/2, ft_vals,   width, label="Fine-tuned (LLaMA Board LoRA)",
                   color="#1565C0", edgecolor="white")

ax.bar_label(bars_base, fmt="{:.2f}", padding=3, fontsize=8)
ax.bar_label(bars_ft,   fmt="{:.2f}", padding=3, fontsize=8)
bars_base[-1].set_color("#FFCC80")
bars_ft[-1].set_color("#E65100")

ax.axhline(1 / len(LABEL_TOKENS), color="grey", linestyle=":", linewidth=1)
ax.text(-0.4, 1 / len(LABEL_TOKENS) + 0.015, "random guess", fontsize=7, color="grey")

ax.set_ylim(0, 1.15)
ax.set_xticks(x)
ax.set_xticklabels([l.replace(" ", "\n") for l in labels_plot], fontsize=9)
ax.set_ylabel("F1 score  /  Accuracy")
ax.set_title("Support ticket router — Baseline vs Fine-tuned (per-class F1 + overall accuracy)")
ax.legend(loc="upper left")
ax.axvline(len(LABEL_TOKENS) - 0.5, color="grey", linestyle="--", linewidth=0.8)
ax.text(len(LABEL_TOKENS) - 0.5 + 0.05, 1.08, "overall", fontsize=8, color="grey")
plt.tight_layout()
plt.savefig("/content/baseline_vs_finetuned.png", dpi=120)
plt.show()

print(f"\nBase accuracy  : {base_acc:.1%}")
print(f"Fine-tuned acc : {ft_acc:.1%}")
print(f"Delta          : {ft_acc - base_acc:+.1%}")
print(f"Random guess   : {1 / len(LABEL_TOKENS):.1%}")

print("\nPer-class F1 change, largest gain first:")
deltas = sorted(((l, ft_f1[l] - base_f1[l]) for l in LABEL_TOKENS),
                key=lambda t: t[1], reverse=True)
for lbl, d in deltas:
    print(f"  {lbl:<20} {base_f1[lbl]:.2f} -> {ft_f1[lbl]:.2f}  ({d:+.2f})")